Download a genome from NCBI. Make an app which samples the genome in the fasta file such that 20000 samples are taken above the genome. Each sample must have a length between 100 and 150 letters.

In [ ]:
import random

FASTA_PATH = "ecoli_cft073.fasta"
N_SAMPLES = 20000
MIN_LEN, MAX_LEN = 100, 150

def read_fasta(path):
    with open(path) as f:
        return "".join(line.strip() for line in f if not line.startswith(">")).upper()

def sample_genome(genome, n_samples, min_len, max_len):
    samples = []
    for _ in range(n_samples):
        length = random.randint(min_len, max_len)
        start = random.randint(0, len(genome) - length)
        samples.append((start, genome[start:start + length]))
    return samples

genome = read_fasta(FASTA_PATH)
print("Genome length:", len(genome))

samples = sample_genome(genome, N_SAMPLES, MIN_LEN, MAX_LEN)

lengths = [len(s) for _, s in samples]
print("Samples:", len(samples))
print("Min and max length:", min(lengths), "and", max(lengths))
print("Mean length: %.1f" % (sum(lengths) / len(lengths)))
print("Average coverage: %.2fx" % (sum(lengths) / len(genome)))
print("First sample:", samples[0][1])

# save the samples as a FASTA file
with open("ecoli_samples.fasta", "w") as out:
    for i, (start, seq) in enumerate(samples):
        out.write(f">read_{i} pos={start} len={len(seq)}\n{seq}\n")

Genome length: 5231428
Samples: 20000
Min and max length: 100 and 150
Mean length: 125.1
Average coverage: 0.48x
First sample: CTGACGCCGAAAGCCAGCAATGGCAATCTGAAGCAGTTCACCATTAACGTGGGACGTGATGGCACAATCCATCAGTTTAGCGCGGTGGAGCAGGACGATCAGCGCAGCAGTTATCAGCTGAAATCCCAGCAAAATGGGGCTGT


Design/ implement an application that is able to use the 20000 samples in order to reconstruct the original genome. The case here is that you consider that you don't have access to the original genome. We have to keep in mind a minimum overlap value.

In [3]:
from collections import defaultdict

SAMPLES_PATH = "ecoli_samples.fasta"
MIN_OVERLAP = 30


def read_reads(path):
    with open(path) as f:
        return [line.strip() for line in f if not line.startswith(">")]


def find_overlaps(reads, min_overlap):
    prefix_index = defaultdict(list)
    for j, r in enumerate(reads):
        prefix_index[r[:min_overlap]].append(j)

    edges = []
    for i, a in enumerate(reads):
        for start in range(1, len(a) - min_overlap + 1):
            for j in prefix_index.get(a[start:start + min_overlap], ()):
                if j != i and reads[j].startswith(a[start:]):
                    edges.append((len(a) - start, i, j))
    return edges


def assemble(reads, min_overlap):
    reads = list(set(reads))
    edges = sorted(find_overlaps(reads, min_overlap), reverse=True)

    nxt, prv = {}, {}
    root = list(range(len(reads)))

    def find(x):
        while root[x] != x:
            root[x] = root[root[x]]
            x = root[x]
        return x

    for ov, a, b in edges:
        if a in nxt or b in prv or find(a) == find(b):
            continue
        nxt[a], prv[b] = (b, ov), a
        root[find(a)] = find(b)

    contigs = []
    for i in range(len(reads)):
        if i in prv:
            continue
        contig, cur = reads[i], i
        while cur in nxt:
            cur, ov = nxt[cur]
            contig += reads[cur][ov:]
        contigs.append(contig)
    return sorted(contigs, key=len, reverse=True)


reads = read_reads(SAMPLES_PATH)
contigs = assemble(reads, MIN_OVERLAP)

contig_lengths = [len(c) for c in contigs]
print("Reads:", len(reads))
print("Contigs:", len(contigs))
print("Longest contig:", contig_lengths[0])
print("Total assembled length:", sum(contig_lengths))
print("Assembled length / genome length: %.1f%%" % (100 * sum(contig_lengths) / len(genome)))

print("Longest 50 contigs found in the original genome:", all(c in genome for c in contigs[:50]))

Reads: 20000
Contigs: 14438
Longest contig: 746
Total assembled length: 2076010
Assembled length / genome length: 39.7%
Longest 50 contigs found in the original genome: False


In [4]:
# save the assembled contigs as a FASTA file (60 letters per line)
with open("ecoli_assembled.fasta", "w") as out:
    for i, contig in enumerate(contigs):
        out.write(f">contig_{i} len={len(contig)}\n")
        for k in range(0, len(contig), 60):
            out.write(contig[k:k + 60] + "\n")

In [5]:
def n50(lengths):
    total, acc = sum(lengths), 0
    for l in sorted(lengths, reverse=True):
        acc += l
        if acc * 2 >= total:
            return l


covered = bytearray(len(genome))
correct, wrong = [], []

for contig in contigs:
    pos = genome.find(contig)
    if pos == -1:
        wrong.append(contig)
    else:
        correct.append(contig)
        covered[pos:pos + len(contig)] = b"\x01" * len(contig)

covered_bases = sum(covered)

print("Original genome length:", len(genome))
print("Assembled contigs:", len(contigs))
print("  exact match in the genome:", len(correct))
print("  not found (misassembled):", len(wrong), "(%.2f%%)" % (100 * len(wrong) / len(contigs)))
print()
print("Total assembled length: %d (%.1f%% of the genome length)" % (
    sum(map(len, contigs)), 100 * sum(map(len, contigs)) / len(genome)))
print("Genome bases covered by correct contigs: %d (%.1f%%)" % (
    covered_bases, 100 * covered_bases / len(genome)))
print("Longest contig:", max(map(len, contigs)))
print("N50 of the assembly:", n50([len(c) for c in contigs]))
print("Longest misassembled contig:", max(map(len, wrong), default=0))

Original genome length: 5231428
Assembled contigs: 14438
  exact match in the genome: 14419
  not found (misassembled): 19 (0.13%)

Total assembled length: 2076010 (39.7% of the genome length)
Genome bases covered by correct contigs: 1979155 (37.8%)
Longest contig: 746
N50 of the assembly: 139
Longest misassembled contig: 470
